# Can a vision-language model (an LLM that reads images) detect seat occupancy?

Small test for the FYP: 23 frames from the laboratory clip (3 seats each = 69 seat decisions) are given to an open vision-language model, **Qwen2.5-VL-3B-Instruct**, running on Colab's free NVIDIA T4 GPU. Its answers are compared with the hand-labelled ground truth and with the YOLO + DeepSORT system on the same frames. The time per frame and the GPU memory are also measured.

The frames are greyscale with the status banner cropped off, so the system's own occupied/vacant colours and counts are hidden. The seat outlines and labels are kept so the model knows which seat is which.

**Before you start**
1. Put `vlm_test.zip` in your Google Drive folder `FYP_colab`.
2. **Runtime → Change runtime type → T4 GPU → Save**.
3. **Runtime → Run all** (about 10–15 minutes, mostly downloading the model).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/FYP_colab'
!rm -rf /content/vlm_test && unzip -q "$DRIVE_DIR/vlm_test.zip" -d /content
%cd /content/vlm_test
!ls frames | head; wc -l frames_gt.csv

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate qwen-vl-utils

## Load the model
If the answers come out empty or garbled, change `DTYPE` to `torch.float32` and run from here again (it still fits on the T4).

In [ ]:
import time

import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
DTYPE = torch.float16
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL, torch_dtype=DTYPE, device_map='cuda')
processor = AutoProcessor.from_pretrained(MODEL, min_pixels=256*28*28, max_pixels=1024*28*28)
n_params = sum(p.numel() for p in model.parameters())
print(f'{MODEL}: {n_params/1e9:.2f} billion parameters')

## Ask the model about every frame

In [ ]:
import csv
import json
import re

from PIL import Image

PROMPT = ('This is a greyscale image from a laboratory camera. Three seat regions are outlined with thin rectangles '
          'and labelled: "cam0_seat_01" (left), "global_desk2" (middle) and "global_desk1" (right). '
          'For each seat, decide whether a person is sitting on the chair in that region. A person walking past or '
          'standing does not count, and bags or jackets on a chair do not count. '
          'Reply with JSON only, exactly in this form: '
          '{"cam0_seat_01": "occupied or vacant", "global_desk2": "occupied or vacant", "global_desk1": "occupied or vacant"}')
def ask(img):
    msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': PROMPT}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors='pt').to('cuda')
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    ans = processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return ans, dt

rows = list(csv.DictReader(open('frames_gt.csv')))
frames = sorted({r['frame'] for r in rows})
ask(Image.open('frames/' + frames[0]).convert('RGB'))      # warm-up, not timed
torch.cuda.reset_peak_memory_stats()
answers, times = {}, []
for f in frames:
    ans, dt = ask(Image.open('frames/' + f).convert('RGB'))
    times.append(dt)
    m = re.search(r'\{.*\}', ans, re.S)
    try: answers[f] = json.loads(m.group(0)) if m else {}
    except Exception: answers[f] = {}
    print(f, f'{dt:.2f} s', ans.replace('\n', ' ')[:150])
peak_gb = torch.cuda.max_memory_allocated() / 1e9

## Results

In [ ]:
def label(v):
    v = str(v).lower()
    return 'occupied' if 'occupied' in v else 'vacant'
for r in rows:
    r['vlm'] = label(answers.get(r['frame'], {}).get(r['seat_id'], 'vacant'))
def metrics(key):
    tp = sum(r[key]=='occupied' and r['ground_truth']=='occupied' for r in rows)
    fp = sum(r[key]=='occupied' and r['ground_truth']=='vacant' for r in rows)
    fn = sum(r[key]!='occupied' and r['ground_truth']=='occupied' for r in rows)
    tn = len(rows) - tp - fp - fn
    p = tp/(tp+fp) if tp+fp else 0; rc = tp/(tp+fn) if tp+fn else 0
    f1 = 2*p*rc/(p+rc) if p+rc else 0
    return dict(accuracy=round((tp+tn)/len(rows),3), precision=round(p,3), recall=round(rc,3), f1=round(f1,3), tp=tp, fp=fp, fn=fn, tn=tn)
unparsed = sum(1 for f in frames if not answers[f])
res = {'model': MODEL, 'dtype': str(DTYPE), 'frames': len(frames), 'seat_decisions': len(rows), 'unparsed_answers': unparsed,
       'vlm': metrics('vlm'), 'yolo_system': metrics('yolo_system'),
       'seconds_per_frame_mean': round(sum(times)/len(times),2), 'seconds_per_frame_max': round(max(times),2),
       'gpu_peak_memory_gb': round(peak_gb,2), 'gpu': torch.cuda.get_device_name(0)}
print(json.dumps(res, indent=2))
print('\nDisagreements with ground truth:')
for r in rows:
    if r['vlm'] != r['ground_truth'] or r['yolo_system'] != r['ground_truth']:
        print(f"  t={r['time_s']:>3}s {r['seat_id']:13s} truth={r['ground_truth']:8s} VLM={r['vlm']:8s} YOLO={r['yolo_system']}")
with open('vlm_results.csv','w',newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
json.dump(res, open('vlm_summary.json','w'), indent=2)

## Save the results

In [ ]:
!cp vlm_results.csv vlm_summary.json "$DRIVE_DIR/"
from google.colab import files

files.download('vlm_summary.json'); files.download('vlm_results.csv')
print('Saved to', DRIVE_DIR)